### Block 0 — Install packages

In [5]:
!pip install -q datasets huggingface_hub anthropic scikit-learn tqdm pandas


### Block 1 — Imports + Hugging Face dataset (FiQA-SA test)

In [6]:
import os
import getpass

from datasets import load_dataset
from huggingface_hub import HfFolder
from tqdm import tqdm
import pandas as pd

from sklearn.metrics import accuracy_score, classification_report


In [7]:
# Hugging Face gated dataset: TheFinAI/flare-fiqasa
DATASET_NAME = "TheFinAI/flare-fiqasa"

# Get HF token (env -> Colab userdata -> manual)
hf_token = HfFolder.get_token() or os.getenv("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not hf_token:
    hf_token = getpass.getpass("Hugging Face token (hf_...): ")

# Load all splits then select test split (235 examples)
ds_all = load_dataset(DATASET_NAME, token=hf_token)
SPLIT = "test"
dataset = ds_all[SPLIT]

print(dataset)
print("Number of examples:", len(dataset))
print("Example 0:", dataset[0])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Hugging Face token (hf_...): ··········


README.md:   0%|          | 0.00/1.13k [00:00<?, ?B/s]

data/train-00000-of-00001-d0f9b6513e12e0(…):   0%|          | 0.00/100k [00:00<?, ?B/s]

data/test-00000-of-00001-faca082021057ac(…):   0%|          | 0.00/35.8k [00:00<?, ?B/s]

data/valid-00000-of-00001-36997935dc03cb(…):   0%|          | 0.00/29.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/750 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/235 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/188 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'query', 'answer', 'text', 'choices', 'gold'],
    num_rows: 235
})
Number of examples: 235
Example 0: {'id': 'fiqasa938', 'query': 'What is the sentiment of the following financial post: Positive, Negative, or Neutral?\nText: $BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash.\nAnswer:', 'answer': 'negative', 'text': '$BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash.', 'choices': ['negative', 'positive', 'neutral'], 'gold': 0}


### Block 2 — Label set + normalization helper

In [8]:
LABELS = ["negative", "neutral", "positive"]

def normalize_prediction(raw: str) -> str:
    """
    Map free-form model output to one of the three labels.
    """
    if raw is None:
        return "neutral"
    text = raw.strip().lower()
    for label in LABELS:
        if label in text:
            return label
    # default fallback
    return "neutral"


### Block 3 — Claude 3 Sonnet client + classifier

In [12]:
import time
from anthropic import Anthropic

# Set your Anthropic API key (prefer env var)
if "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key (sk-...): ")

client_claude = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

MODEL_ID_CLAUDE = "claude-sonnet-4-20250514"

SYSTEM_PROMPT_CLAUDE = (
    "You are a financial sentiment classifier.\n"
    "Given a single sentence from a financial news article or microblog,\n"
    "classify its overall sentiment from the perspective of an investor as\n"
    "exactly one of: negative, neutral, or positive.\n"
    "Respond with only one word: negative, neutral, or positive."
)

def classify_with_claude_sonnet(sentence: str,
                                max_retries: int = 6,
                                sleep_base: float = 1.0) -> str:
    """
    Call Claude 3 Sonnet once and return a normalized 3-way label.
    """
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client_claude.messages.create(
                model=MODEL_ID_CLAUDE,
                max_tokens=16,
                temperature=0.0,
                system=SYSTEM_PROMPT_CLAUDE,
                messages=[{"role": "user", "content": sentence}],
            )
            # Anthropic messages API: text is in resp.content[0].text
            raw = resp.content[0].text if resp.content else ""
            return normalize_prediction(raw)
        except Exception as e:
            last_err = e
            wait = sleep_base * (2 ** attempt)
            print(f"[Claude Sonnet] Error on attempt {attempt+1}/{max_retries}: {last_err}")
            time.sleep(wait)

    print("[Claude Sonnet] Failed after retries, returning 'neutral'.")
    return "neutral"


### Block 4 — Evaluation loop on FiQA-SA test split

In [14]:
y_true = []
y_pred = []

print(f"Evaluating model: {MODEL_ID_CLAUDE} on {len(dataset)} FiQA-SA test examples...\n")

for example in tqdm(dataset, total=len(dataset)):
    sentence = example["text"]
    true_label = example["answer"].lower().strip()
    pred_label = classify_with_claude_sonnet(sentence)

    y_true.append(true_label)
    y_pred.append(pred_label)

print("Total examples:", len(dataset))
print("Got predictions for:", len(y_pred))


Evaluating model: claude-sonnet-4-20250514 on 235 FiQA-SA test examples...



100%|██████████| 235/235 [09:22<00:00,  2.39s/it]

Total examples: 235
Got predictions for: 235


### Block 5 — Metrics + simple error analysis

In [15]:
# Overall metrics
accuracy = accuracy_score(y_true, y_pred)
print(f"\nOverall accuracy: {accuracy:.4f}\n")

print("Classification report:")
print(classification_report(y_true, y_pred, labels=LABELS))

# Build DataFrame for inspection
df = pd.DataFrame({
    "text": [ex["text"] for ex in dataset],
    "gold": y_true,
    "pred": y_pred,
})

print("\nFirst 10 predictions:")
print(df.head(10).to_string(index=False))

# Show some errors
errors = df[df["gold"] != df["pred"]]
print(f"\nNumber of errors: {len(errors)}")
print("Some error examples:")
print(errors.head(20).to_string(index=False))



Overall accuracy: 0.7702

Classification report:
              precision    recall  f1-score   support

    negative       0.84      0.88      0.86        76
     neutral       0.22      0.61      0.32        18
    positive       0.99      0.73      0.84       141

    accuracy                           0.77       235
   macro avg       0.68      0.74      0.67       235
weighted avg       0.88      0.77      0.81       235


First 10 predictions:
                                                                                                                            text     gold     pred
                                                     $BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash. negative negative
                                                                         Legal & General share price: Finance chief to step down negative negative
                                                                 Kingfisher share price slides on cost to